In [1]:
# Cell 1: Check GPU
!nvidia-smi

Tue Jun  2 03:53:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Cell 2: Clone your repo
!git clone https://github.com/dcyforjob2020/ai-interview-coach.git
%cd ai-interview-coach


fatal: destination path 'ai-interview-coach' already exists and is not an empty directory.
/content/ai-interview-coach


In [3]:
# Cell 3: Install dependencies
!pip install -q -U \
  "transformers==4.46.3" \
  accelerate \
  peft \
  datasets \
  bitsandbytes \
  trl \
  sentencepiece \
  "protobuf<6"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 161.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 119.7 MB/s eta 0:00:00


In [4]:
import torch
import transformers
import peft
import trl
import bitsandbytes

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)


torch: 2.11.0+cu128
cuda available: True
transformers: 4.46.3
peft: 0.19.1
trl: 0.17.0


In [ ]:
# Cell 4: Optional, login to Hugging Face
# Qwen models are usually public, but login helps if Colab hits access/rate limits.
from huggingface_hub import notebook_login
notebook_login()


In [5]:
# Cell 5: Build the SFT dataset from preference pairs
!python train/sft/build_sft_dataset.py \
  --input train/preference_pairs.jsonl \
  --output train/sft/sft_dataset.jsonl


Read: 2000 records from train/preference_pairs.jsonl
Wrote: 2000 SFT records to train/sft/sft_dataset.jsonl


In [6]:
# Cell 6: Quick sanity check of the SFT dataset
import json

path = "train/sft/sft_dataset.jsonl"

with open(path, "r", encoding="utf-8") as f:
    first = json.loads(next(f))

print(first.keys())
print(first["id"])
print(first["messages"][0])
print(first["messages"][1]["content"][:500])
print(first["messages"][2]["content"][:500])


dict_keys(['id', 'messages', 'source', 'judge_model', 'winner'])
train_0001
{'role': 'system', 'content': 'You are a precise and helpful AI interview coach for CS students. Give feedback directly to the student using second person. Identify what is correct, what is missing or incorrect, and how to improve the answer.'}
Question:
What is the purpose of using analysis packages in software development?

Reference answer:
The purpose of using analysis packages is to facilitate the organization, understanding, and management of the analysis model, allowing developers and analysts to focus on specific parts of the system more effectively and manage the complexity of the project.

Student answer:
Analysis packages help organize the analysis model so everyone can work on different parts of the system without getting o
You provided a great explanation for the purpose of using analysis packages in software development. Your answer highlights the key benefits of organizing the analysis model, whi

In [8]:
# Cell 7: Train SFT with QLoRA
# For T4 GPU, 1024 max seq length is safer.
# If you have L4/A100, you can try 2048.
!python train/sft/train_sft_qlora.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --data train/sft/sft_dataset.jsonl \
  --output-dir train/model/sft_adapter \
  --max-seq-length 1024 \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum 8 \
  --learning-rate 1e-4 \
  --save-steps 200 \
  --logging-steps 10


Loading checkpoint shards:   0% 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100% 2/2 [00:04<00:00,  2.27s/it]
{'loss': 1.6941, 'grad_norm': 0.8450790047645569, 'learning_rate': 9.998314826517563e-05, 'num_tokens': 29769.0, 'mean_token_accuracy': 0.6034737218171358, 'epoch': 0.04}
{'loss': 1.0302, 'grad_norm': 0.7301179766654968, 'learning_rate': 9.939452940908626e-05, 'num_tokens': 59812.0, 'mean_token_accuracy': 0.7100991457700729, 'epoch': 0.08}
{'loss': 0.8082, 'grad_norm': 0.5054270029067993, 'learning_rate': 9.797464868072488e-05, 'num_tokens': 89847.0, 'mean_token_accuracy': 0.7525695331394673, 'epoch': 0.12}
{'loss': 0.7978, 'grad_norm': 0.5083761215209961, 'learning_rate': 9.574740129129767e-05, 'num_tokens': 121607.0, 'mean_to

In [9]:
# Cell 8: Zip the trained adapter so you can download it
!zip -r sft_adapter.zip train/model/sft_adapter


  adding: train/model/sft_adapter/ (stored 0%)
  adding: train/model/sft_adapter/training_args.bin (deflated 53%)
  adding: train/model/sft_adapter/checkpoint-250/ (stored 0%)
  adding: train/model/sft_adapter/checkpoint-250/training_args.bin (deflated 53%)
  adding: train/model/sft_adapter/checkpoint-250/rng_state.pth (deflated 26%)
  adding: train/model/sft_adapter/checkpoint-250/optimizer.pt (deflated 13%)
  adding: train/model/sft_adapter/checkpoint-250/special_tokens_map.json (deflated 69%)
  adding: train/model/sft_adapter/checkpoint-250/tokenizer.json (deflated 81%)
  adding: train/model/sft_adapter/checkpoint-250/vocab.json (deflated 61%)
  adding: train/model/sft_adapter/checkpoint-250/adapter_config.json (deflated 59%)
  adding: train/model/sft_adapter/checkpoint-250/adapter_model.safetensors (deflated 8%)
  adding: train/model/sft_adapter/checkpoint-250/tokenizer_config.json (deflated 83%)
  adding: train/model/sft_adapter/checkpoint-250/trainer_state.json (deflated 75%)
  a

In [10]:
# Cell 9: Download the adapter
from google.colab import files
files.download("sft_adapter.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
!zip -r sft_adapter_final_only.zip \
  train/model/sft_adapter/adapter_config.json \
  train/model/sft_adapter/adapter_model.safetensors \
  train/model/sft_adapter/tokenizer.json \
  train/model/sft_adapter/tokenizer_config.json \
  train/model/sft_adapter/special_tokens_map.json \
  train/model/sft_adapter/vocab.json \
  train/model/sft_adapter/merges.txt \
  train/model/sft_adapter/README.md \
  train/model/sft_adapter/added_tokens.json \
  train/model/sft_adapter/training_args.bin

  adding: train/model/sft_adapter/adapter_config.json (deflated 59%)
  adding: train/model/sft_adapter/adapter_model.safetensors (deflated 8%)
  adding: train/model/sft_adapter/tokenizer.json (deflated 81%)
  adding: train/model/sft_adapter/tokenizer_config.json (deflated 83%)
  adding: train/model/sft_adapter/special_tokens_map.json (deflated 69%)
  adding: train/model/sft_adapter/vocab.json (deflated 61%)
  adding: train/model/sft_adapter/merges.txt (deflated 57%)
  adding: train/model/sft_adapter/README.md (deflated 65%)
  adding: train/model/sft_adapter/added_tokens.json (deflated 67%)
  adding: train/model/sft_adapter/training_args.bin (deflated 53%)


In [12]:
from google.colab import files
files.download("sft_adapter_final_only.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
from huggingface_hub import notebook_login
notebook_login()


In [16]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "Hank122222222/ai-interview-coach-sft-adapter"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=False,
    exist_ok=True,
)

api.upload_folder(
    folder_path="train/model/sft_adapter",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload SFT LoRA adapter",
    ignore_patterns=[
        "checkpoint-*",
        "optimizer.pt",
        "scheduler.pt",
        "rng_state.pth",
        "trainer_state.json",
    ],
)

print("Uploaded to:", f"https://huggingface.co/{repo_id}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  559kB /  120MB            

  ...ft_adapter/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter/training_args.bin:   1%|1         |  63.0B / 6.03kB            

Uploaded to: https://huggingface.co/Hank122222222/ai-interview-coach-sft-adapter
